<a href="https://colab.research.google.com/github/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN_02/blob/main/DBTT_03_dataset_dataloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sổ tay 03: Lớp Dataset và DataLoader

**Dự án:** Nhận dạng tham số động cơ PMSM: mạng MLP nhận dạng Ld, Lq theo (id, iq); Rs tính đại số từ Ld, Lq đã nhận dạng.

**Nguyên tắc:** sổ tay chạy độc lập; mỗi lần chạy đều thiết lập lại môi trường từ đầu. Mã được viết, kiểm thử bằng dữ liệu mẫu, xuất thành tệp `.py` rồi đẩy lên GitHub theo bốn bước của PRD.

### Sơ đồ khối của sổ tay (BPMN 2.0, ISO/IEC 19510)



*Hình: Sơ đồ khối quá trình của sổ tay 03*

## Cell 1 – Thiết lập môi trường

In [2]:
# Cell 1 - Thiết lập môi trường từ đầu: gắn Drive, nhận mã truy cập, sao chép kho mã
import os  # Thao tác đường dẫn và tệp
import sys  # Điều chỉnh đường dẫn nạp mô-đun
import subprocess  # Chạy lệnh git và pip
from google.colab import drive, userdata  # Gắn Drive và đọc bí mật của Colab
drive.mount('/content/drive')  # Gắn Google Drive vào phiên Colab
GITHUB_USER = 'maivu18082005-hub'  # Tài khoản GitHub chứa kho mã
REPO_NAME = 'Parameter_Identification_of_PMSM_using_PINN_02'  # Tên kho mã
REPO_DIR = f'/content/{REPO_NAME}'  # Thư mục kho trên máy Colab
TOKEN = userdata.get('GITHUB_TOKEN')  # Mã truy cập lấy từ Colab Secrets
REMOTE = f'https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'  # Địa chỉ kho kèm mã truy cập


def run(cmd, check=True):  # Hàm chạy lệnh và che mã truy cập trong đầu ra
    r = subprocess.run(cmd, capture_output=True, text=True)  # Chạy lệnh, thu đầu ra
    out = (r.stdout + r.stderr).replace(TOKEN, '***')  # Thay mã truy cập bằng dấu sao
    if check and r.returncode != 0:  # Nếu lệnh lỗi và yêu cầu kiểm tra
        raise RuntimeError(out)  # Báo lỗi kèm đầu ra đã che
    return out  # Trả về đầu ra


def push_files(paths, message):  # Hàm đẩy tệp lên GitHub
    run(['git', '-C', REPO_DIR, 'config', 'user.name', GITHUB_USER])  # Đặt tên người commit
    run(['git', '-C', REPO_DIR, 'config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com'])  # Đặt email người commit
    run(['git', '-C', REPO_DIR, 'add'] + paths)  # Thêm các tệp vào vùng chờ
    if not run(['git', '-C', REPO_DIR, 'status', '--porcelain']).strip():  # Nếu không có thay đổi
        print('Không có thay đổi để đẩy')  # Thông báo
        return  # Kết thúc hàm
    run(['git', '-C', REPO_DIR, 'commit', '-m', message])  # Tạo commit
    run(['git', '-C', REPO_DIR, 'branch', '-M', 'main'])  # Đảm bảo nhánh là main
    run(['git', '-C', REPO_DIR, 'pull', '--rebase', REMOTE, 'main'], check=False)  # Đồng bộ trước khi đẩy
    print(run(['git', '-C', REPO_DIR, 'push', REMOTE, 'HEAD:main']))  # Đẩy lên GitHub


if os.path.isdir(os.path.join(REPO_DIR, '.git')):  # Nếu kho đã có trên máy
    run(['git', '-C', REPO_DIR, 'pull', REMOTE, 'main'], check=False)  # Cập nhật kho
else:  # Nếu chưa có
    run(['git', 'clone', REMOTE, REPO_DIR])  # Sao chép kho từ GitHub
sys.path.insert(0, REPO_DIR)  # Cho phép nạp mô-đun từ kho
TESTS_OK = False  # Cờ cho phép đẩy mã, chỉ bật sau khi kiểm thử đạt
print('Kho mã sẵn sàng tại', REPO_DIR)  # Thông báo hoàn tất

req = os.path.join(REPO_DIR, 'requirements.txt')  # Tệp yêu cầu thư viện
if os.path.exists(req):  # Nếu tệp tồn tại
    run([sys.executable, '-m', 'pip', 'install', '-q', '-r', req])  # Cài thư viện
import torch  # Thư viện học sâu
from src.utils.common import load_config, set_seed, get_paths  # Tiện ích dùng chung
cfg = load_config(os.path.join(REPO_DIR, 'configs', 'config.yaml'))  # Nạp cấu hình
set_seed(cfg['project']['seed'])  # Cố định hạt giống
PATHS = get_paths(cfg)  # Đường dẫn tuyệt đối trên Drive
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'  # Chọn thiết bị tính toán
print('Thiết bị:', DEVICE, '| Torch:', torch.__version__)  # In thông tin môi trường

Mounted at /content/drive
Kho mã sẵn sàng tại /content/Parameter_Identification_of_PMSM_using_PINN_02
Thiết bị: cpu | Torch: 2.14.1+cpu


## Cell 2 – Nạp dữ liệu đã xử lý và thống kê nhãn
Kiểm tra các tệp `*_processed.csv.gz` và `stats.json` do sổ tay 02 tạo ra.

In [3]:
# Cell 2 - Kiểm tra điều kiện đầu vào từ sổ tay 02
import json  # Đọc thống kê JSON
import numpy as np  # Thư viện số học mảng
import pandas as pd  # Thư viện bảng dữ liệu
need = [f.replace('.csv', '_processed.csv.gz') for f in cfg['data']['raw_files']] + ['stats.json']  # Danh sách tệp cần có
missing = [n for n in need if not os.path.exists(os.path.join(PATHS['processed'], n))]  # Tệp còn thiếu
assert not missing, f'Chạy sổ tay 02 trước. Thiếu: {missing}'  # Dừng nếu thiếu
with open(os.path.join(PATHS['processed'], 'stats.json'), encoding='utf-8') as fh:  # Mở thống kê nhãn
    stats = json.load(fh)  # Đọc thống kê
print('Khoảng nhãn mở rộng:', {k: (round(v['lo'], 6), round(v['hi'], 6)) for k, v in stats.items()})  # In khoảng nhãn

Khoảng nhãn mở rộng: {'Rs': (0.027062, 0.053023), 'Ld': (0.0009, 0.001598), 'Lq': (0.001013, 0.001726)}


## Cell 3 – Viết `src/data/dataset.py`
Lớp `PMSMDataset` trả về đầu vào (id, iq) theo ampe và nhãn (Ld, Lq) theo henry; hàm `build_dataloaders` chia tập theo tệp. Rs không là nhãn của MLP; Rs nhãn chỉ dùng khi đánh giá ở sổ tay 06.

In [4]:
%%writefile /content/Parameter_Identification_of_PMSM_using_PINN_02/src/data/dataset.py
"""Lớp Dataset và hàm dựng DataLoader cho MLP nhận dạng Ld, Lq theo (id, iq)."""  # Mô tả mô-đun
import os  # Thao tác đường dẫn
import numpy as np  # Thư viện số học mảng
import pandas as pd  # Thư viện bảng dữ liệu
import torch  # Thư viện học sâu
from torch.utils.data import Dataset, DataLoader  # Lớp cơ sở và bộ nạp dữ liệu

INPUT_COLS = ["Id", "Iq"]  # Đầu vào của mạng (A)
LABEL_COLS = ["Ld", "Lq"]  # Nhãn của mạng (H)


class PMSMDataset(Dataset):  # Lớp tập dữ liệu
    """Tập dữ liệu PMSM: đầu vào (id, iq) và nhãn (Ld, Lq) theo đơn vị SI."""  # Mô tả lớp

    def __init__(self, frames):  # Hàm khởi tạo
        """Nhận danh sách bảng đã xử lý và dựng các tensor."""  # Mô tả hàm
        df = pd.concat(frames, ignore_index=True)  # Gộp các bảng
        self.x = torch.tensor(df[INPUT_COLS].to_numpy(dtype=np.float32))  # Tensor đầu vào
        self.y = torch.tensor(df[LABEL_COLS].to_numpy(dtype=np.float32))  # Tensor nhãn
        self.flags = df[["is_transient"]].to_numpy(dtype=bool)  # Cờ vận hành (quá độ), chỉ để thống kê

    def __len__(self):  # Số mẫu
        """Trả về số mẫu của tập dữ liệu."""  # Mô tả hàm
        return self.x.shape[0]  # Số hàng của tensor đầu vào

    def __getitem__(self, i):  # Lấy một mẫu
        """Trả về từ điển gồm x, y của mẫu thứ i."""  # Mô tả hàm
        return {"x": self.x[i], "y": self.y[i]}  # Gói mẫu


def load_processed(processed_dir, names):  # Hàm nạp các tệp đã xử lý
    """Nạp danh sách tệp đã xử lý theo tên tệp thô."""  # Mô tả hàm
    frames = []  # Danh sách bảng
    for n in names:  # Duyệt từng tên tệp
        stem = os.path.splitext(n)[0]  # Bỏ phần mở rộng
        frames.append(pd.read_csv(os.path.join(processed_dir, f"{stem}_processed.csv.gz")))  # Đọc tệp đã xử lý
    return frames  # Trả về danh sách bảng


def build_dataloaders(cfg, processed_dir, num_workers=0):  # Hàm dựng bộ nạp dữ liệu
    """Dựng DataLoader cho tập huấn luyện, kiểm định và kiểm thử theo cấu hình."""  # Mô tả hàm
    sp, bs = cfg["data"]["split"], cfg["train"]["batch_size"]  # Cách chia tập và kích thước lô
    sets = {k: PMSMDataset(load_processed(processed_dir, sp[k])) for k in ("train", "val", "test")}  # Dựng ba tập
    g = torch.Generator().manual_seed(cfg["project"]["seed"])  # Bộ sinh ngẫu nhiên cố định
    loaders = {  # Từ điển bộ nạp
        "train": DataLoader(sets["train"], batch_size=bs, shuffle=True, generator=g, num_workers=num_workers, drop_last=False),  # Trộn mẫu khi huấn luyện
        "val": DataLoader(sets["val"], batch_size=bs, shuffle=False, num_workers=num_workers),  # Giữ thứ tự khi kiểm định
        "test": DataLoader(sets["test"], batch_size=bs, shuffle=False, num_workers=num_workers),  # Giữ thứ tự khi kiểm thử
    }  # Kết thúc từ điển
    return sets, loaders  # Trả về tập và bộ nạp

Writing /content/Parameter_Identification_of_PMSM_using_PINN_02/src/data/dataset.py


## Cell 4 – Kiểm thử đơn vị
Kiểm tra chia tập đúng Data1-3 / Data4 / Data5, đầu vào đúng hai cột (id, iq) và nhãn đúng hai cột (Ld, Lq), kích thước, kiểu dữ liệu, giá trị hữu hạn, không rò rỉ giữa các tập, tính tái lập của trộn mẫu; in dải đầu vào và tỷ lệ nhãn kiểm định, kiểm thử nằm ngoài cận đầu ra của mạng.

In [5]:
# Cell 4 - Kiểm thử đơn vị cho dataset.py
import importlib  # Nạp lại mô-đun vừa ghi
import torch  # Thư viện học sâu
import src.data.dataset as ds  # Mô-đun cần kiểm thử
importlib.reload(ds)  # Dùng bản mới ghi
assert ds.INPUT_COLS == ['Id', 'Iq'] and ds.LABEL_COLS == ['Ld', 'Lq'], 'Đầu vào phải là (Id, Iq), nhãn phải là (Ld, Lq)'  # Kiểm tra cột
sets, loaders = ds.build_dataloaders(cfg, PATHS['processed'])  # Dựng tập và bộ nạp
sp = cfg['data']['split']  # Cách chia tập
assert sp['train'] == ['Data1.csv', 'Data2.csv', 'Data3.csv'] and sp['val'] == ['Data4.csv'] and sp['test'] == ['Data5.csv'], 'Chia tập phải là Data1-3 / Data4 / Data5'  # Kiểm tra chia tập
all_names = sp['train'] + sp['val'] + sp['test']  # Tên tệp của cả ba tập
assert len(all_names) == len(set(all_names)), 'Một tệp xuất hiện ở nhiều tập (rò rỉ)'  # Kiểm tra không giao nhau
for k, d in sets.items():  # Duyệt từng tập
    assert d.x.shape[1] == 2 and d.y.shape[1] == 2, f'Sai số chiều tập {k}'  # Kiểm tra số chiều
    assert d.x.dtype == torch.float32 and d.y.dtype == torch.float32, 'Sai kiểu dữ liệu'  # Kiểm tra kiểu
    assert torch.isfinite(d.x).all() and torch.isfinite(d.y).all(), f'Có giá trị không hữu hạn ở tập {k}'  # Kiểm tra hữu hạn
    assert len(d) > 0, f'Tập {k} rỗng'  # Kiểm tra không rỗng
    assert (d.y > 0).all(), f'Nhãn Ld, Lq phải dương ở tập {k}'  # Điện cảm phải dương
b = next(iter(loaders['train']))  # Lấy một lô huấn luyện
assert set(b) == {'x', 'y'} and b['x'].shape == (min(cfg['train']['batch_size'], len(sets['train'])), 2) and b['y'].shape == b['x'].shape, 'Sai kích thước lô'  # Kiểm tra kích thước lô
f0 = ds.load_processed(PATHS['processed'], sp['train'][:1])[0]  # Tệp huấn luyện đầu tiên
assert np.array_equal(sets['train'].x[:len(f0)].numpy(), f0[['Id', 'Iq']].to_numpy(np.float32)) and np.array_equal(sets['train'].y[:len(f0)].numpy(), f0[['Ld', 'Lq']].to_numpy(np.float32)), 'Cột x, y phải khớp bảng đã xử lý'  # Kiểm tra căn chỉnh
_, l2 = ds.build_dataloaders(cfg, PATHS['processed'])  # Dựng lần hai
assert torch.equal(next(iter(l2['train']))['x'], b['x']), 'Trộn mẫu chưa tái lập'  # Kiểm tra tái lập
i_base = cfg['motor']['i_rated_peak']  # Dòng cơ sở
x = sets['train'].x.numpy()  # Đầu vào huấn luyện
print('Dải đầu vào theo đơn vị tương đối:', {n: (round(float(x[:, i].min() / i_base), 3), round(float(x[:, i].max() / i_base), 3)) for i, n in enumerate(ds.INPUT_COLS)})  # In dải đầu vào
for k, d in sets.items():  # Duyệt từng tập
    print(k, 'số mẫu:', len(d), '| tỷ lệ quá độ:', round(float(d.flags[:, 0].mean()), 4))  # In thống kê tập
for k in ('val', 'test'):  # Nhãn ngoài cận đầu ra làm mạng không thể đạt tới nhãn
    y = sets[k].y.numpy()  # Nhãn của tập
    out = {n: round(float(np.mean((y[:, i] < stats[n]['lo']) | (y[:, i] > stats[n]['hi'])) * 100), 3) for i, n in enumerate(ds.LABEL_COLS)}  # Tỷ lệ ngoài cận (%)
    print(f'Tỷ lệ nhãn tập {k} nằm ngoài cận đầu ra của mạng (%):', out, '' if max(out.values()) == 0 else '<- CẢNH BÁO: tăng data.bound_margin hoặc xem lại phân bố dữ liệu')  # In kết quả
TESTS_OK = True  # Bật cờ cho phép đẩy mã
print('Toàn bộ kiểm thử đơn vị đạt')  # Thông báo

Dải đầu vào theo đơn vị tương đối: {'Id': (-0.768, 0.065), 'Iq': (-1.177, 1.238)}
train số mẫu: 899940 | tỷ lệ quá độ: 0.2223
val số mẫu: 299980 | tỷ lệ quá độ: 0.1
test số mẫu: 299980 | tỷ lệ quá độ: 0.0667
Tỷ lệ nhãn tập val nằm ngoài cận đầu ra của mạng (%): {'Ld': 0.0, 'Lq': 0.0} 
Tỷ lệ nhãn tập test nằm ngoài cận đầu ra của mạng (%): {'Ld': 0.0, 'Lq': 0.0} 
Toàn bộ kiểm thử đơn vị đạt


## Cell 5 – Bước 4: đẩy lên GitHub

In [6]:
# Cell cuối - Bước 4: đẩy tệp đã kiểm thử đạt lên GitHub
assert TESTS_OK, 'Chưa đạt toàn bộ kiểm thử, không được đẩy mã'  # Chặn đẩy mã khi kiểm thử chưa đạt
push_files(['src/data/dataset.py'], 'Thêm dataset.py')  # Đẩy các tệp lên nhánh main

To https://github.com/maivu18082005-hub/Parameter_Identification_of_PMSM_using_PINN_02.git
   db72d3c..e125dcb  HEAD -> main

